# Downloading necessary datasets

**Note**: Remember to create download-env conda environemnt located at `/projappl/your_project_name/env/download-env.yml` and make it available as Jupyter kernel.

In [ ]:
from pathlib import Path

PROJECT_NAME = "your_project_name"  # set once; all paths below derive from it
PROJECT = Path(f"/scratch/{PROJECT_NAME}")
DATASETS = PROJECT / "datasets"

GRID_DIR = DATASETS / "kaggle_lipnet"
LRS3_DIR = DATASETS / "lrs3"
MUSAN_DIR = DATASETS / "musan_noise"

GRID_DIR.mkdir(parents=True, exist_ok=True)
LRS3_DIR.mkdir(parents=True, exist_ok=True)
MUSAN_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset directory:", DATASETS)

Dataset directory: \scratch\your_project_name\datasets


In [ ]:
import os
KAGGLE_TOKEN = os.environ["KAGGLE_TOKEN"]
HF_TOKEN = os.environ["HF_TOKEN"]

### GRID dataset download

In [ ]:
import kagglehub

kagglehub.login()

In [ ]:
import os

# Tell KaggleHub to use scratch instead of your home directory
os.environ["KAGGLEHUB_CACHE"] = str(GRID_DIR)
print("Kaggle cache:", os.environ["KAGGLEHUB_CACHE"])

In [ ]:
import kagglehub

GRID_path = kagglehub.dataset_download(
    "jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet"
)

print("KaggleHub downloaded dataset to:")
print(GRID_path)

### LRS3 dataset download

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN) 

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="TheNHz/ellipsis-lrs3-raw",
    repo_type="dataset",
    allow_patterns=[
        "ainncy/trainval.tar.gz",
        "landmarks/LRS3_landmarks.zip",
    ],
    local_dir=str(LRS3_DIR),
)

print("LRS3 downloaded to:")
print(path)

#### LRS3 test videos (original-resolution)

The real LRS3 test videos (`lrs3/test/<video_id>/<clip_id>.mp4` + `.txt`) come from
`ballooncat/lrsss`, a ~145GB split archive (7 parts). Only the `lrs3/test/*` part of it is needed.
The parts are downloaded to scratch (NOT the default cache, which lands in `$HOME`); make sure
the target has ~150GB free.

In [ ]:
from huggingface_hub import hf_hub_download

PARTS = [f"Lrs3.tar.gz{i:02d}" for i in range(7)]
LRS3_TEST_ARCHIVE_DIR = LRS3_DIR / "ballooncat_raw"  # where the 7 archive parts are saved
LRS3_TEST_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

for part in PARTS:
    if (LRS3_TEST_ARCHIVE_DIR / part).exists():
        print(f"{part} already downloaded, skipping.")
        continue

    print(f"Downloading {part}...")
    local_path = hf_hub_download(
        repo_id="ballooncat/lrsss",
        filename=part,
        repo_type="dataset",
        local_dir=str(LRS3_TEST_ARCHIVE_DIR),  # real file, on scratch, not home
    )
    print(f"{part} saved to {local_path}\n")

print("All parts downloaded to:", LRS3_TEST_ARCHIVE_DIR)

Extract only the test split (optional, once all parts are downloaded). The parts are one gzip
stream split into chunks, so they are concatenated in order into a single `tar`. `-C` must come
BEFORE the `--wildcards` pattern (it only affects arguments after it), and the target directory
must already exist. This creates `<datasets>/lrs3/test/`.

In [ ]:
import subprocess

part_paths = " ".join(f"'{LRS3_TEST_ARCHIVE_DIR / part}'" for part in PARTS)
subprocess.run(
    f"cat {part_paths} | tar -xzf - -C '{DATASETS}' --wildcards 'lrs3/test/*'",
    shell=True, check=True,
)
print("Extracted to:", LRS3_DIR / "test")

### MUSAN dataset download

In [ ]:
import os

# Tell KaggleHub to use scratch instead of your home directory
os.environ["KAGGLEHUB_CACHE"] = str(MUSAN_DIR)
print("Kaggle cache:", os.environ["KAGGLEHUB_CACHE"])

In [ ]:
import kagglehub

MUSAN_path = kagglehub.dataset_download("nhattruongdev/musan-noise")

print("KaggleHub downloaded dataset to:")
print(MUSAN_path)

### Disk usage

In [ ]:
import subprocess

subprocess.run(["du", "-sh", str(DATASETS)])

### Compressing LRS3 download for Allas transfer (optional)